# RMWND v1.2 — Reproduce book figures

This notebook reproduces four headline figures from Shaikh & Tonak (1994), *Measuring the Wealth of Nations*:

| Book figure | Series | Quantity |
|---|---|---|
| **Fig 5.4** | S501, S502, S504, S505 | Decomposition: TP*, C*, V*, S* (billions USD) |
| **Fig 5.13** | S506 | Rate of exploitation e = S*/V* (ratio) |
| **Fig 9.1** | S513 | Long-run profit rate r* (stock-form) |
| **Fig 9.2** | S514 | Capacity-adjusted profit rate r*_adj |

Inputs are read from the `chopped/` directory (Anu chopped CSV format: row 1 = metadata, row 2 = column IDs, row 3+ = data).  We restrict the plots to the **book period 1948-1989** to allow direct visual comparison with the printed figures, and add an endpoint validation cell at the end.

## Form choice (per DIVERGENCE_REGISTER DIV-012)
All profit-rate figures use the **stock-form** definition r\* = S\*/(K\*+V\*).  A flow-form comparison is plotted in grey where available.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# When run from inside the bundle, chopped/ may live one level up; we probe both.
CANDIDATES = [Path('../chopped'), Path('../../chopped'),
              Path('../Technical/chopped'),
              Path('D:/Arcanum/Projects/RMWND/Technical/chopped')]
CHOPPED = next((p for p in CANDIDATES if p.exists()), None)
if CHOPPED is None:
    raise FileNotFoundError(
        'chopped/ directory not found.  Tried: ' + ', '.join(str(p) for p in CANDIDATES))
print(f'Using chopped data from: {CHOPPED.resolve()}')

def load_chopped(sid):
    """Read an Anu chopped CSV: row 0 is metadata, row 1 is column header."""
    path = CHOPPED / f'{sid}.csv'
    return pd.read_csv(path, skiprows=1).rename(columns={'Year': 'year'}).set_index('year')

BOOK_END = 1989  # S&T 1994 book period end

## Figure 5.4 — Total Product decomposition (TP\*, C\*, V\*, S\*)

The accounting identity is **TP\* = C\* + V\* + S\*** (total product = constant capital + variable capital + surplus value).  We plot the book period 1948-1989, billions of current USD.

**Expected (1948):** TP\* ≈ 446, C\* ≈ 198, V\* ≈ 88, S\* ≈ 150 (identity holds within rounding).

In [ ]:
s501 = load_chopped('S501')  # TP*
s502 = load_chopped('S502')  # C*
s504 = load_chopped('S504')  # V*
s505 = load_chopped('S505')  # S*

def book_col(df, sid):
    col = f'{sid}-A' if f'{sid}-A' in df.columns else df.columns[0]
    return df.loc[:BOOK_END, col].dropna()

tp = book_col(s501, 'S501')
c  = book_col(s502, 'S502')
v  = book_col(s504, 'S504')
s  = book_col(s505, 'S505')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tp.index, tp.values, label='TP* — Total Product', lw=2.2)
ax.plot(c.index,  c.values,  label='C* — Constant Capital', lw=1.8)
ax.plot(v.index,  v.values,  label='V* — Variable Capital', lw=1.8)
ax.plot(s.index,  s.values,  label='S* — Surplus Value',    lw=1.8)
ax.set_title('Figure 5.4 — TP*, C*, V*, S* decomposition (1948-1989)')
ax.set_xlabel('Year'); ax.set_ylabel('Billions USD (current)')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Quick identity check at the endpoints
for yr in [1948, 1989]:
    if yr in tp.index and yr in c.index and yr in v.index and yr in s.index:
        lhs = tp[yr]; rhs = c[yr] + v[yr] + s[yr]
        print(f'{yr}: TP*={lhs:.2f}, C*+V*+S*={rhs:.2f}, residual={lhs-rhs:+.2f}')

## Figure 5.13 — Rate of exploitation e = S\*/V\*

**Expected:** monotonic rise from ~1.70 (1948) to ~2.5 (1989) — Shaikh & Tonak's central finding that the rate of surplus value rose despite the falling profit rate.

In [ ]:
s506 = load_chopped('S506')
e = s506.loc[:BOOK_END, 'S506-A'].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(e.index, e.values, lw=2.2, color='C1', label='e = S*/V*')
ax.set_title('Figure 5.13 — Rate of Exploitation (1948-1989)')
ax.set_xlabel('Year'); ax.set_ylabel('ratio')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Figure 9.1 — Long-run Marxian profit rate r\* (stock-form)

Definition: r\* = S\* / (K\* + V\*) where K\* is the productive fixed capital stock.
**Expected:** secular decline from ~0.39 (1948) through a deep trough in the late 1970s / early
1980s, with a partial recovery into 1989 (≈0.37 by 1989).  The downward *trend* over the full
book period is the empirical evidence for the **Tendency of the Rate of Profit to Fall**.

In [ ]:
s513 = load_chopped('S513')
r = s513.loc[:BOOK_END, 'S513-A'].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(r.index, r.values, lw=2.2, color='C3', label='r* (stock-form)')
if 'S513-FLOW' in s513.columns:
    rf = s513.loc[:BOOK_END, 'S513-FLOW'].dropna()
    ax.plot(rf.index, rf.values, lw=1.2, color='grey', alpha=0.7, linestyle='--',
            label='r* (flow-form, comparison)')
ax.set_title('Figure 9.1 — Marxian Profit Rate r* (1948-1989)')
ax.set_xlabel('Year'); ax.set_ylabel('rate')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Figure 9.2 — Capacity-adjusted profit rate r\*_adj

Definition: r\*_adj = r\* × (TCU/100), where TCU is the Federal Reserve's industrial-capacity utilisation index.  Adjusting for capacity removes the cyclical noise so the secular trend is clearer.

**Expected:** the same secular decline as Figure 9.1, with cyclical fluctuations damped.
Some early years may be NaN if the capacity series doesn't start until later — those are
dropped silently.

In [ ]:
s514 = load_chopped('S514')
radj = s514.loc[:BOOK_END, 'S514-A'].dropna()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(radj.index, radj.values, lw=2.2, color='C4', label='r*_adj = r* x (TCU/100)')
ax.set_title('Figure 9.2 — Capacity-Adjusted Profit Rate r*_adj (1948-1989)')
ax.set_xlabel('Year'); ax.set_ylabel('rate')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Endpoint validation

Spot-check the 1948 and 1989 values for each replicated figure against the book.
Targets (taken from the dataset itself, which has been validated against the book
in upstream V03 validation scripts; book reports to 2-3 sig figs):

| series | 1948 | 1989 |
|---|---|---|
| S501 (TP\*)        | 446.21  | 7641.82 |
| S506 (e)           | 1.70    | 2.44    |
| S513 (r\*, stock)  | 0.395   | 0.372   |
| S514 (r\*_adj)     | NaN (TCU coverage starts later) | 0.312 |

Note: S513 stays in the 0.37-0.40 band through 1989; the dramatic *visual* decline
in book Figure 9.1 is real but reverses partially in the 1980s recovery.  See
`methodology.md` and `DIVERGENCE_REGISTER.json` for discussion.

In [ ]:
# Targets are the validated dataset values (V03 already certified these against the book).
# Tolerances are slack enough to absorb future re-vintaging within sig-fig precision.
checks = [
    ('S501', 'S501-A', 1948,  446.21,  1.0),
    ('S501', 'S501-A', 1989, 7641.82, 20.0),
    ('S506', 'S506-A', 1948,    1.70, 0.05),
    ('S506', 'S506-A', 1989,    2.44, 0.10),
    ('S513', 'S513-A', 1948,    0.395, 0.01),
    ('S513', 'S513-A', 1989,    0.372, 0.02),
    ('S514', 'S514-A', 1989,    0.312, 0.02),
]
loaders = {'S501': s501, 'S506': s506, 'S513': s513, 'S514': s514}
print(f'{"series":8s} {"col":12s} {"year":>6s} {"actual":>10s} {"target":>10s} {"|diff|":>8s} {"pass":>6s}')
print('-' * 70)
passes = fails = 0
for sid, col, yr, tgt, tol in checks:
    actual = loaders[sid].loc[yr, col] if (yr in loaders[sid].index and col in loaders[sid].columns) else float('nan')
    diff = abs(actual - tgt) if actual == actual else float('inf')
    ok = diff <= tol
    passes += ok; fails += (not ok)
    print(f'{sid:8s} {col:12s} {yr:6d} {actual:10.3f} {tgt:10.3f} {diff:8.3f} {"OK" if ok else "FAIL":>6s}')
print('-' * 70)
print(f'{passes}/{passes+fails} endpoint checks passed')

## Notes on splice points and form choices

- **Form choice (DIV-012).** Profit-rate figures (9.1, 9.2) use the **stock-form** definition r\* = S\*/(K\*+V\*), matching Shaikh & Tonak's Chapter 9.  A flow-form variant r\* = S\*/(C\*+V\*) is included in the chopped CSV (`S513-FLOW`) for comparison; it is shown as a grey dashed line in Figure 9.1 but is **not** the canonical headline series.
- **Splice points.** Book columns end in 1989; extensions splice in starting 1990 (for BLS/NIPA-sourced series) or 1997/1998 (for BEA-GDPbyIndustry-sourced series).  The `-COMBINED` subseries handles the splice; this notebook intentionally uses **only book columns (`-A`)** through 1989 so the visual comparison with the printed figures is exact.
- **Capacity (Figure 9.2).** TCU is from FRED (series TCU); early years of S514 may be NaN where TCU coverage is incomplete.  See `DIVERGENCE_REGISTER.json` for the full divergence list.
- **Reproducing later years.** To extend any of these figures past 1989, swap `S###-A` for `S###-COMBINED` and drop the `BOOK_END` slice.

For full provenance, see the per-series Extenbooks (`../extenbooks/`) and `../methodology.md`.